# 01 — WT | Exploration and event inspection

Purpose: environment setup; inspect source trial markers; inspect one pilot neuron from MOs.

**No mass preprocessing occurs here.** The long-running WT pipeline is in `02_process_and_cache.ipynb`. Statistics and heatmaps are in `03_analysis_heatmaps.ipynb`.

Drive root: `MyDrive/Reanalysis/Our_Analysis/Behavior_aligned`. Spontaneous results remain under `Our_Analysis/Spontaneous`.


In [ ]:
# Setup: run once per fresh Colab runtime.
%pip -q install one-api ibllib iblatlas pandas numpy matplotlib

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
from one.api import ONE

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned')
META = ROOT / '02_session_metadata'
INVENTORY_PATH = META / 'mouse_probe_inventory_recovered.csv'

if not INVENTORY_PATH.is_file():
    raise FileNotFoundError(
        f'Missing recovered inventory: {INVENTORY_PATH}. '
        'Do not recreate it automatically; check the project folder.'
    )

inventory = pd.read_csv(INVENTORY_PATH)
if inventory.empty or not {'eid', 'pid', 'subject', 'genotype'}.issubset(inventory.columns):
    raise ValueError('Recovered inventory is empty or has unexpected columns.')

# ONE uses a local download cache; source analysis files are not modified.
one = ONE(
    base_url='https://openalyx.internationalbrainlab.org',
    username='intbrainlab',
    password='international',
    silent=True
)

print('Insertions:', inventory['pid'].nunique())
print('Sessions:', inventory['eid'].nunique())
print('Mice:', inventory['subject'].nunique())
display(inventory.groupby('genotype').agg(
    insertions=('pid', 'nunique'), sessions=('eid', 'nunique'),
    mice=('subject', 'nunique')
))

## Pilot behavioral session: NYU-58

In [ ]:
# Load all behavior objects once.
EID = '91cc517d-628d-402f-a8a4-6bd239b97b36'
SESSION_MATCH = inventory.loc[inventory['eid'].astype(str) == EID]
if SESSION_MATCH.empty:
    raise ValueError('Pilot EID is not in the recovered inventory.')
print('Pilot session:', EID)
display(SESSION_MATCH[['subject', 'genotype', 'eid', 'probe', 'pid']].head())

trial_obj = one.load_object(EID, 'trials', collection='alf')
wheel_obj = one.load_object(EID, 'wheel', collection='alf')
moves_obj = one.load_object(EID, 'wheelMoves', collection='alf')

# Assemble scalar event markers, including 2-column trial intervals.
if 'stimOn_times' not in trial_obj or 'intervals' not in trial_obj:
    raise ValueError('Expected trial events are missing in the selected session.')
intervals = np.asarray(trial_obj['intervals'], dtype=float)
if intervals.ndim != 2 or intervals.shape[1] != 2:
    raise ValueError(f'Unexpected trial intervals shape: {intervals.shape}')
n_trials = intervals.shape[0]

trial_cols = {}
for name, values in trial_obj.items():
    arr = np.asarray(values)
    if arr.ndim == 1 and len(arr) == n_trials:
        trial_cols[name] = arr
trial_cols['trial_start'] = intervals[:, 0]
trial_cols['trial_end'] = intervals[:, 1]
trials = pd.DataFrame(trial_cols)

wheel_time = np.asarray(wheel_obj['timestamps'], dtype=float)
wheel_pos = np.asarray(wheel_obj['position'], dtype=float)
wheel_moves = np.asarray(moves_obj['intervals'], dtype=float)

assert wheel_time.ndim == wheel_pos.ndim == 1
assert len(wheel_time) == len(wheel_pos)
assert wheel_moves.ndim == 2 and wheel_moves.shape[1] == 2
assert np.all(np.diff(wheel_time) >= 0), 'Wheel timestamps are not ordered.'

print(f'Trials: {len(trials):,}')
print(f'Wheel position samples: {len(wheel_time):,}')
print(f'Wheel movement intervals: {len(wheel_moves):,}')
print('Available trial columns:', ', '.join(trials.columns))

## Trial timestamps and markers

In [ ]:
# Event audit; no files are created or overwritten.
EVENT_DESCRIPTIONS = {
    'trial_start': 'Trial start', 'trial_end': 'Trial end',
    'stimOn_times': 'Stimulus onset',
    'stimOff_times': 'Stimulus offset',
    'stimOnTrigger_times': 'Stimulus-on trigger',
    'stimOffTrigger_times': 'Stimulus-off trigger',
    'goCue_times': 'Go cue', 'goCueTrigger_times': 'Go cue trigger',
    'firstMovement_times': 'First detected movement',
    'response_times': 'Response', 'feedback_times': 'Feedback'
}

available_names = [key for key in EVENT_DESCRIPTIONS if key in trials.columns]
event_table = pd.DataFrame([
    {'Event': key, 'Meaning': EVENT_DESCRIPTIONS[key],
     'Available trials': int(pd.to_numeric(trials[key], errors='coerce').notna().sum()),
     'Missing trials': int(pd.to_numeric(trials[key], errors='coerce').isna().sum())}
    for key in available_names
])
print('EVENT TIMESTAMP AVAILABILITY')
display(event_table)

order = ['stimOn_times', 'goCue_times', 'firstMovement_times',
         'response_times', 'feedback_times', 'stimOff_times', 'trial_end']
order = [key for key in order if key in trials.columns]
preview = trials.iloc[:10][['trial_start'] + order].copy()
for name in order:
    preview[name] = pd.to_numeric(preview[name], errors='coerce') - preview['trial_start']
preview.insert(0, 'trial', np.arange(1, len(preview) + 1))
print('FIRST 10 TRIALS — SECONDS RELATIVE TO TRIAL START')
display(preview.round(3))

if {'stimOn_times', 'goCue_times'}.issubset(trials.columns):
    delta = (trials['goCue_times'] - trials['stimOn_times']).dropna()
    print(f'Go cue minus stimulus onset: median {delta.median():.4f} s')
if {'firstMovement_times', 'stimOn_times'}.issubset(trials.columns):
    early = (trials['firstMovement_times'] < trials['stimOn_times']).fillna(False)
    print(f'Trials with first movement before stimulus onset: {early.sum()} / {len(trials)}')

## Visual event timelines

In [ ]:
# Visualization only; no automatic figure export or file overwrites.
EVENTS = [
    ('stimOn_times', 'Stim ON', '#008C80'),
    ('goCue_times', 'Go cue', '#C08A25'),
    ('firstMovement_times', 'First movement', '#E06A35'),
    ('response_times', 'Response', '#A94B88'),
    ('feedback_times', 'Feedback', '#777777'),
    ('stimOff_times', 'Stim OFF', '#2865B0'),
]

def event_time(trial, name):
    try:
        return float(trial[name]) if name in trial.index else np.nan
    except (ValueError, TypeError):
        return np.nan


def plot_trial(trial_index):
    """Plot one trial in three clearly labeled, time-aligned panels."""
    trial = trials.iloc[trial_index]
    t0, t1 = event_time(trial, 'trial_start'), event_time(trial, 'trial_end')
    if not (np.isfinite(t0) and np.isfinite(t1) and t1 > t0):
        print(f'Trial {trial_index+1}: missing or invalid boundaries')
        return
    dur = t1 - t0
    fig, (ax_wheel, ax_bars, ax_events) = plt.subplots(
        3, 1, figsize=(12, 8), sharex=True,
        gridspec_kw={'height_ratios': [2.2, 1.0, 2.6]},
        layout='constrained'
    )

    # Wheel displacement relative to the first wheel sample in the trial.
    mask = (wheel_time >= t0) & (wheel_time <= t1)
    if mask.sum() > 1:
        relative = wheel_pos[mask] - wheel_pos[mask][0]
        ax_wheel.plot(wheel_time[mask] - t0, relative, lw=1.7, color='#222222')
    ax_wheel.set_ylabel('Wheel displacement (rad)')
    ax_wheel.set_title(f'WT pilot | trial {trial_index + 1} | duration {dur:.2f} s',
                       fontweight='bold')

    # Stimulus display interval, independently of wheel motion.
    stim_on, stim_off = event_time(trial, 'stimOn_times'), event_time(trial, 'stimOff_times')
    if np.isfinite(stim_on) and np.isfinite(stim_off):
        a, b = max(stim_on, t0) - t0, min(stim_off, t1) - t0
        if b > a:
            ax_bars.broken_barh([(a, b-a)], (1.45, .4), facecolors='#66C2A5')

    # Actual movement intervals, not just firstMovement_times.
    overlaps = wheel_moves[(wheel_moves[:, 0] < t1) & (wheel_moves[:, 1] > t0)]
    for start, end in overlaps:
        a, b = max(float(start), t0) - t0, min(float(end), t1) - t0
        if b > a:
            ax_bars.broken_barh([(a, b-a)], (.35, .4), facecolors='#F5A06A')
    ax_bars.set_ylim(0, 2.15)
    ax_bars.set_yticks([.55, 1.65], ['Wheel movement', 'Stimulus visible'])

    # Distinct rows for event markers; shared time axis.
    observed = [(label, event_time(trial, key) - t0, color)
                for key, label, color in EVENTS
                if np.isfinite(event_time(trial, key))
                and t0 <= event_time(trial, key) <= t1]
    for i, (label, sec, color) in enumerate(observed):
        y = len(observed) - i - 1
        ax_events.scatter(sec, y, s=85, c=color, zorder=3)
        ax_events.text(sec + .025, y + .13, f'{sec:.3f} s',
                       fontsize=9, color=color, fontweight='bold')
    ax_events.set_yticks(range(len(observed)),
                         [label for label, _, _ in reversed(observed)])
    ax_events.set_ylim(-.6, len(observed)-.35)
    ax_events.set_xlabel('Seconds from trial start')

    for ax in (ax_wheel, ax_bars, ax_events):
        ax.set_xlim(0, dur)
        ax.grid(axis='x', alpha=.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    plt.show()
    plt.close(fig)

# Visualize the same first three pilot trials (no disk writes).
for trial_index in [0, 1, 2]:
    plot_trial(trial_index)

## Example MOs neuron overlay (optional)

In [ ]:

# ============================================================
# SINGLE MOs NEURON — SPIKES ALIGNED TO BEHAVIOR
# One mouse, one session, one neuron, multiple trials
# Clean figure with dedicated title and legend areas
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from brainbox.io.one import SpikeSortingLoader

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")
OUT = ROOT / "05_figures" / "WT" / "Pilot_timelines"
(OUT / "PNG").mkdir(parents=True, exist_ok=True)
(OUT / "PDF").mkdir(parents=True, exist_ok=True)
METADATA_DIR = ROOT / "04_results" / "WT"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

N_TRIALS = 8
BIN_S = 0.05
SMOOTH_SIGMA_S = 0.15

# ============================================================
# 1. SELECT A WT MOs NEURON
# ============================================================

neurons = pd.read_csv(
    ROOT / "03_processed_data" / "WT" / "WT_all_neurons.csv"
)

mos = neurons.loc[
    neurons["region"].eq("MOs")
].copy()

if mos.empty:
    raise RuntimeError("No WT MOs neurons found.")

session_counts = (
    mos.groupby("eid")
    .size()
    .sort_values(ascending=False)
)

selected_eid = None

for eid in session_counts.index:

    try:
        names = [str(x) for x in one.list_datasets(str(eid))]

        required = [
            "_ibl_trials.table.pqt",
            "_ibl_wheel.timestamps.npy",
            "_ibl_wheelMoves.intervals.npy"
        ]

        if all(any(x in n for n in names) for x in required):
            selected_eid = str(eid)
            break

    except Exception:
        continue

if selected_eid is None:
    raise RuntimeError("No suitable WT MOs session found.")

# ============================================================
# 2. LOAD BEHAVIORAL DATA
# ============================================================

trial_obj = one.load_object(
    selected_eid, "trials", collection="alf"
)

wheel_obj = one.load_object(
    selected_eid, "wheel", collection="alf"
)

move_obj = one.load_object(
    selected_eid, "wheelMoves", collection="alf"
)

intervals = np.asarray(trial_obj["intervals"], dtype=float)

trial_data = {
    "trial_start": intervals[:, 0],
    "trial_end": intervals[:, 1]
}

EVENT_KEYS = [
    "stimOn_times",
    "stimOff_times",
    "goCue_times",
    "firstMovement_times",
    "response_times",
    "feedback_times"
]

for key in EVENT_KEYS:
    if key in trial_obj:
        values = np.asarray(trial_obj[key])
        if values.ndim == 1 and len(values) == len(intervals):
            trial_data[key] = values

tr = pd.DataFrame(trial_data)

wt = np.asarray(wheel_obj["timestamps"], dtype=float)
wp = np.asarray(wheel_obj["position"], dtype=float)
wm = np.asarray(move_obj["intervals"], dtype=float)

# ============================================================
# 3. LOAD ONE MOs NEURON
# ============================================================

session_neurons = (
    mos.loc[mos["eid"].astype(str) == selected_eid]
    .sort_values("n_spikes", ascending=False)
)

selected = None
spike_times = None

for pid, units in session_neurons.groupby("pid", sort=False):

    try:
        loader = SpikeSortingLoader(
            pid=str(pid), one=one
        )

        spikes, clusters, channels = loader.load_spike_sorting()

        spike_clusters = np.asarray(spikes["clusters"])
        all_times = np.asarray(spikes["times"], dtype=float)

        for _, unit in units.iterrows():

            cluster_id = int(unit["cluster_id"])
            mask = spike_clusters == cluster_id

            if np.count_nonzero(mask) >= 100:

                selected = unit
                spike_times = np.sort(all_times[mask])
                break

        if selected is not None:
            break

    except Exception as exc:
        print("Skipped probe:", str(pid)[:8], str(exc)[:100])

if selected is None:
    raise RuntimeError("Could not load an MOs neuron.")

print("=" * 65)
print("SELECTED NEURON")
print("=" * 65)
print("Mouse:", selected["subject"])
print("Genotype:", selected["genotype"])
print("Session:", selected_eid)
print("Probe:", selected["pid"])
print("Cluster:", selected["cluster_id"])
print("Region:", selected["region"])
print("Total spikes:", len(spike_times))

# ============================================================
# 4. SELECT CONSECUTIVE TRIALS
# ============================================================

n_window = min(N_TRIALS, len(tr))
start_idx = None

for i in range(len(tr) - n_window + 1):

    a = float(tr.iloc[i]["trial_start"])
    b = float(tr.iloc[i + n_window - 1]["trial_end"])

    if (
        np.isfinite(a)
        and np.isfinite(b)
        and a >= spike_times[0]
        and b <= spike_times[-1]
    ):
        start_idx = i
        break

if start_idx is None:
    raise RuntimeError("No complete trial block found.")

block = tr.iloc[
    start_idx:start_idx + n_window
].copy()

T0 = float(block["trial_start"].iloc[0])
T1 = float(block["trial_end"].iloc[-1])

selected_spikes = (
    spike_times[
        (spike_times >= T0) &
        (spike_times < T1)
    ] - T0
)

duration = T1 - T0

# ============================================================
# 5. COMPUTE VISUALIZATION FIRING RATE
# ============================================================

edges = np.arange(0, duration, BIN_S)
edges = np.append(edges, duration)

counts, _ = np.histogram(
    selected_spikes,
    bins=edges
)

widths = np.diff(edges)
centers = (edges[:-1] + edges[1:]) / 2

rate = counts / widths

smoothed_rate = gaussian_filter1d(
    rate,
    sigma=SMOOTH_SIGMA_S / BIN_S
)

# ============================================================
# 6. FIGURE — DEDICATED TITLE AND LEGEND SPACE
# ============================================================

stim_color = "#64B99C"
move_color = "#E59B68"

fig = plt.figure(figsize=(17, 10.8))

gs = fig.add_gridspec(
    4, 1,
    height_ratios=[0.65, 1.1, 1.0, 1.5],
    hspace=0.16,
    left=0.085,
    right=0.985,
    top=0.96,
    bottom=0.075
)

# Dedicated legend axis — no overlap with the data.
ax_header = fig.add_subplot(gs[0])
ax_header.axis("off")

ax_spikes = fig.add_subplot(gs[1])
ax_rate = fig.add_subplot(gs[2], sharex=ax_spikes)
ax_wheel = fig.add_subplot(gs[3], sharex=ax_spikes)

axes = [ax_spikes, ax_rate, ax_wheel]

# ------------------------------------------------------------
# TITLE
# ------------------------------------------------------------

ax_header.text(
    0.5, 0.82,
    f"WT | MOs single neuron | "
    f"{selected['subject']} | "
    f"Cluster {int(selected['cluster_id'])}",
    transform=ax_header.transAxes,
    ha="center",
    va="center",
    fontsize=17,
    fontweight="bold"
)

# ------------------------------------------------------------
# LEGEND — INSIDE HEADER, OUTSIDE DATA PANELS
# ------------------------------------------------------------

legend_items = [
    Line2D(
        [0], [0],
        color="#242424",
        linewidth=2,
        label="Neuron spikes"
    ),
    Line2D(
        [0], [0],
        color="#355C95",
        linewidth=2,
        label="Smoothed firing rate"
    ),
    Patch(
        facecolor=stim_color,
        alpha=0.45,
        label="Visual stimulus visible"
    ),
    Patch(
        facecolor=move_color,
        alpha=0.45,
        label="Wheel movement"
    ),
    Line2D(
        [0], [0],
        color="#A57F29",
        linestyle="--",
        linewidth=1.5,
        label="Go cue"
    ),
    Line2D(
        [0], [0],
        color="#A84A82",
        linestyle="-.",
        linewidth=1.5,
        label="Response"
    )
]

ax_header.legend(
    handles=legend_items,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.05),
    ncol=3,
    frameon=False,
    fontsize=10,
    columnspacing=2.5,
    handlelength=2.8
)

# ============================================================
# 7. NEURON SPIKE TRAIN
# ============================================================

ax_spikes.vlines(
    selected_spikes,
    0, 1,
    color="#242424",
    linewidth=0.7
)

ax_spikes.set_ylim(0, 1.12)
ax_spikes.set_yticks([])
ax_spikes.set_ylabel(
    "Single-neuron\nspikes",
    fontsize=11
)

# ============================================================
# 8. SMOOTHED FIRING RATE
# ============================================================

ax_rate.plot(
    centers,
    smoothed_rate,
    color="#355C95",
    linewidth=1.7,
    zorder=3
)

ax_rate.set_ylabel(
    "Firing rate\n(Hz)",
    fontsize=11
)

# ============================================================
# 9. WHEEL DISPLACEMENT
# ============================================================

wheel_mask = (
    (wt >= T0) &
    (wt <= T1)
)

wheel_t = wt[wheel_mask] - T0
wheel_p = wp[wheel_mask]

if len(wheel_p) > 0:
    ax_wheel.plot(
        wheel_t,
        wheel_p - wheel_p[0],
        color="#303030",
        linewidth=1.3,
        zorder=3
    )

ax_wheel.set_ylabel(
    "Wheel displacement\n(rad)",
    fontsize=11
)

ax_wheel.set_xlabel(
    "Time from displayed block start (s)",
    fontsize=12
)

# ============================================================
# 10. OVERLAY VISUAL STIMULUS WINDOWS
# ============================================================

for _, row in block.iterrows():

    s_on = row.get("stimOn_times", np.nan)
    s_off = row.get("stimOff_times", np.nan)

    if np.isfinite(s_on) and np.isfinite(s_off):

        left = max(float(s_on), T0) - T0
        right = min(float(s_off), T1) - T0

        if right > left:

            for ax in axes:
                ax.axvspan(
                    left,
                    right,
                    color=stim_color,
                    alpha=0.17,
                    zorder=0
                )

    # Trial start markers
    boundary = float(row["trial_start"]) - T0

    for ax in axes:
        ax.axvline(
            boundary,
            color="#BBBBBB",
            linestyle=":",
            linewidth=0.9,
            zorder=1
        )

# ============================================================
# 11. OVERLAY ACTUAL WHEEL MOVEMENTS
# ============================================================

for a, b in wm:

    left = max(float(a), T0) - T0
    right = min(float(b), T1) - T0

    if right <= left:
        continue

    # Highlight movement intervals on wheel trace.
    ax_wheel.axvspan(
        left,
        right,
        color=move_color,
        alpha=0.25,
        zorder=0
    )

    # Orange horizontal markers above spike raster.
    ax_spikes.plot(
        [left, right],
        [1.045, 1.045],
        color="#D77D43",
        linewidth=3,
        solid_capstyle="butt",
        zorder=4
    )

# ============================================================
# 12. GO CUE AND RESPONSE MARKERS
# ============================================================

for _, row in block.iterrows():

    for event, color, style in [
        ("goCue_times", "#A57F29", "--"),
        ("response_times", "#A84A82", "-.")
    ]:

        value = row.get(event, np.nan)

        if np.isfinite(value) and T0 <= value <= T1:

            ax_wheel.axvline(
                float(value) - T0,
                color=color,
                linestyle=style,
                linewidth=1.2,
                alpha=0.8,
                zorder=4
            )

# ============================================================
# 13. FINAL FORMATTING
# ============================================================

for ax in axes:

    ax.set_xlim(0, duration)
    ax.grid(axis="x", alpha=0.12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

ax_spikes.tick_params(labelbottom=False)
ax_rate.tick_params(labelbottom=False)

# ============================================================
# 14. SAVE FIGURE
# ============================================================

output = "WT_MOs_single_neuron_behavior_overlay"

fig.savefig(
    OUT / "PNG" / f"{output}.png",
    dpi=350,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    OUT / "PDF" / f"{output}.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ============================================================
# 15. SAVE RECORDING METADATA
# ============================================================

metadata = pd.DataFrame([{
    "subject": selected["subject"],
    "genotype": selected["genotype"],
    "eid": selected_eid,
    "pid": selected["pid"],
    "cluster_id": int(selected["cluster_id"]),
    "region": selected["region"],
    "first_trial": start_idx + 1,
    "last_trial": start_idx + n_window,
    "start_s": T0,
    "end_s": T1,
    "duration_s": duration,
    "displayed_spikes": len(selected_spikes)
}])

metadata.to_csv(
    METADATA_DIR / "WT_MOs_single_neuron_behavior_overlay_metadata.csv",
    index=False
)

print("\nFIGURES SAVED:", OUT / "PNG" / f"{output}.png", OUT / "PDF" / f"{output}.pdf")
display(metadata)
